<a href="https://colab.research.google.com/github/naeembashir63/dlnp/blob/main/DLNP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install datasets
!pip install sentence-transformers
!pip install faiss-cpu
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 111.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 148.5 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
from datasets import load_dataset

dataset = load_dataset("hotpot_qa", "fullwiki", split="train[:100]")

# Example
sample = dataset[0]
print(sample["question"])
print(sample["answer"])
print(sample["supporting_facts"])

Which magazine was started first Arthur's Magazine or First for Women?
Arthur's Magazine
{'title': ["Arthur's Magazine", 'First for Women'], 'sent_id': [0, 0]}


In [ ]:
def get_titles(sample):
    return list(set(sample["supporting_facts"]["title"]))

titles = get_titles(sample)
print(titles)

['First for Women', "Arthur's Magazine"]


In [ ]:
def get_context(sample):
    context_list = []
    for title_list, sentences_list in zip(sample["context"]["title"], sample["context"]["sentences"]):
        context_list.extend(sentences_list)
    return context_list

In [ ]:
def chunk_text(text, size=200):
    words = text.split()
    return [" ".join(words[i:i+size]) for i in range(0, len(words), size)]

docs = []
for sample in dataset:
    contexts = get_context(sample)
    for c in contexts:
        docs.extend(chunk_text(c, 200))

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = model.encode(docs)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
import faiss
import numpy as np

dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(np.array(embeddings))

In [ ]:
def retrieve(query, k=3):
    q_emb = model.encode([query])
    distances, indices = index.search(q_emb, k)
    return [docs[i] for i in indices[0]]

In [ ]:
from transformers import pipeline

qa_pipeline = pipeline("question-answering")

def answer_question(question):
    contexts = retrieve(question, k=3)
    context = " ".join(contexts)

    result = qa_pipeline(question=question, context=context)
    return result["answer"]

No model was supplied, defaulted to distilbert/distilbert-base-cased-distilled-squad and revision 564e9b5.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

In [ ]:
for i in range(5):
    q = dataset[i]["question"]
    gt = dataset[i]["answer"]

    pred = answer_question(q)

    print("Q:", q)
    print("Pred:", pred)
    print("GT:", gt)
    print("------")

Q: Which magazine was started first Arthur's Magazine or First for Women?
Pred: First for Women
GT: Arthur's Magazine
------
Q: The Oberoi family is part of a hotel company that has a head office in what city?
Pred: Delhi
GT: Delhi
------
Q: Musician and satirist Allie Goertz wrote a song about the "The Simpsons" character Milhouse, who Matt Groening named after who?
Pred: President Richard Nixon
GT: President Richard Nixon
------
Q:  What nationality was James Henry Miller's wife?
Pred: Australian
GT: American
------
Q: Cadmium Chloride is slightly soluble in this chemical, it is also called what?
Pred: CdCl
GT: alcohol
------


In [ ]:
def exact_match(pred, gt):
    return int(pred.lower() == gt.lower())

In [ ]:
# ==============================
# 1. LOAD DATASET
# ==============================
from datasets import load_dataset

print("\n[STEP 1] Loading dataset...")
dataset = load_dataset("hotpot_qa", "fullwiki", split="train[:100]")

print("[INFO] Dataset size:", len(dataset))
print("[INFO] First question:", dataset[0]["question"])
print("[INFO] First answer:", dataset[0]["answer"])
print("------")


# ==============================
# 2. FILTER CONTEXT
# ==============================
print("\n[STEP 2] Preparing context (supporting facts)...")

def get_context(sample):
    titles = set(sample["supporting_facts"]["title"])
    context_list = []

    for title, sentences in zip(sample["context"]["title"], sample["context"]["sentences"]):
        if title in titles:
            context_list.extend(sentences)

    return context_list


sample_context = get_context(dataset[0])
print("[DEBUG] Supporting context sentences:", len(sample_context))
print("[DEBUG] Example context:", sample_context[:2])
print("------")


# ==============================
# 3. CHUNKING
# ==============================
print("\n[STEP 3] Chunking text...")

def chunk_text(text):
    return [text]

print("[INFO] Chunking method: sentence-level (no split)")
print("------")


# ==============================
# 4. BUILD DOCUMENT STORE
# ==============================
print("\n[STEP 4] Building document store...")

docs = []

for idx, sample in enumerate(dataset):
    contexts = get_context(sample)
    for c in contexts:
        docs.extend(chunk_text(c))

    if idx % 20 == 0:
        print(f"[PROGRESS] Processed {idx} samples")

print("[INFO] Total documents/chunks:", len(docs))
print("[DEBUG] Sample doc:", docs[0][:100])
print("------")


# ==============================
# 5. EMBEDDINGS
# ==============================
print("\n[STEP 5] Encoding documents into embeddings...")

from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(docs, normalize_embeddings=True)

print("[INFO] Embedding dimension:", embeddings.shape[1])
print("[INFO] Total embeddings:", embeddings.shape[0])
print("------")


# ==============================
# 6. FAISS INDEX
# ==============================
print("\n[STEP 6] Building FAISS index...")

import faiss
import numpy as np

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(np.array(embeddings))

print("[INFO] FAISS index size:", index.ntotal)
print("------")


# ==============================
# 7. LOAD RERANKER
# ==============================
print("\n[STEP 7] Loading reranker...")

from sentence_transformers import CrossEncoder

reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

print("[INFO] Reranker loaded")
print("------")


# ==============================
# 8. RETRIEVAL + RERANKING
# ==============================
print("\n[STEP 8] Defining retrieval function...")

def retrieve(query, k=5):
    print("\n[RETRIEVAL DEBUG]")
    print("Query:", query)

    # Step 1: initial retrieval (top 20)
    q_emb = model.encode([query], normalize_embeddings=True)
    distances, indices = index.search(q_emb, 20)

    candidate_docs = [docs[i] for i in indices[0]]

    print("[DEBUG] Retrieved (before rerank):", len(candidate_docs))

    # Step 2: reranking
    pairs = [(query, doc) for doc in candidate_docs]
    scores = reranker.predict(pairs)

    ranked = sorted(zip(candidate_docs, scores), key=lambda x: x[1], reverse=True)

    # Step 3: take top-k
    top_docs = [doc for doc, _ in ranked[:k]]

    print("[DEBUG] Top docs after reranking:")
    for i, doc in enumerate(top_docs):
        print(f"{i+1}. {doc[:120]}...")

    return top_docs


# ==============================
# 9. QA MODEL
# ==============================
print("\n[STEP 9] Loading QA model...")

from transformers import pipeline

qa_pipeline = pipeline(
    "question-answering",
    model="deepset/roberta-base-squad2"
)

print("[INFO] QA model loaded")
print("------")


# ==============================
# 10. ANSWER FUNCTION
# ==============================
print("\n[STEP 10] Defining answer function...")

def answer_question(question):
    contexts = retrieve(question, k=5)
    context = " ".join(contexts)

    print("[DEBUG] Context length:", len(context))

    result = qa_pipeline(question=question, context=context)

    print("[DEBUG] QA output:", result)

    return result["answer"]


# ==============================
# 11. NORMALIZATION
# ==============================
def normalize(text):
    return text.lower().strip()


# ==============================
# 12. F1 SCORE
# ==============================
def compute_f1(pred, gt):
    pred_tokens = normalize(pred).split()
    gt_tokens = normalize(gt).split()

    common = set(pred_tokens) & set(gt_tokens)

    if len(common) == 0:
        return 0.0

    precision = len(common) / len(pred_tokens)
    recall = len(common) / len(gt_tokens)

    return 2 * (precision * recall) / (precision + recall)


# ==============================
# 13. RUN TEST
# ==============================
print("\n[STEP 13] Running evaluation...\n")

total_f1 = 0

for i in range(100):
    print("\n==============================")
    print(f"[TEST CASE {i+1}]")

    q = dataset[i]["question"]
    gt = dataset[i]["answer"]

    print("[QUESTION]", q)
    print("[GROUND TRUTH]", gt)

    pred = answer_question(q)

    f1 = compute_f1(pred, gt)
    total_f1 += f1

    print("\n[PREDICTED]", pred)
    print("[F1 SCORE]", round(f1, 3))
    print("==============================")

avg_f1 = total_f1 / 100
print("\n🔥 FINAL AVERAGE F1:", round(avg_f1, 3))


[STEP 1] Loading dataset...
[INFO] Dataset size: 100
[INFO] First question: Which magazine was started first Arthur's Magazine or First for Women?
[INFO] First answer: Arthur's Magazine
------

[STEP 2] Preparing context (supporting facts)...
[DEBUG] Supporting context sentences: 7
[DEBUG] Example context: ["Arthur's Magazine (1844–1846) was an American literary periodical published in Philadelphia in the 19th century.", ' Edited by T.S. Arthur, it featured work by Edgar A. Poe, J.H. Ingraham, Sarah Josepha Hale, Thomas G. Spear, and others.']
------

[STEP 3] Chunking text...
[INFO] Chunking method: sentence-level (no split)
------

[STEP 4] Building document store...
[PROGRESS] Processed 0 samples
[PROGRESS] Processed 20 samples
[PROGRESS] Processed 40 samples
[PROGRESS] Processed 60 samples
[PROGRESS] Processed 80 samples
[INFO] Total documents/chunks: 667
[DEBUG] Sample doc: Arthur's Magazine (1844–1846) was an American literary periodical published in Philadelphia in the 1
------

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[INFO] Embedding dimension: 384
[INFO] Total embeddings: 667
------

[STEP 6] Building FAISS index...
[INFO] FAISS index size: 667
------

[STEP 7] Loading reranker...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[INFO] Reranker loaded
------

[STEP 8] Defining retrieval function...

[STEP 9] Loading QA model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaForQuestionAnswering LOAD REPORT from: deepset/roberta-base-squad2
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[INFO] QA model loaded
------

[STEP 10] Defining answer function...

[STEP 13] Running evaluation...


[TEST CASE 1]
[QUESTION] Which magazine was started first Arthur's Magazine or First for Women?
[GROUND TRUTH] Arthur's Magazine

[RETRIEVAL DEBUG]
Query: Which magazine was started first Arthur's Magazine or First for Women?
[DEBUG] Retrieved (before rerank): 20
[DEBUG] Top docs after reranking:
1. First for Women is a woman's magazine published by Bauer Media Group in the USA....
2. First for Women is a woman's magazine published by Bauer Media Group in the USA....
3. Arthur's Magazine (1844–1846) was an American literary periodical published in Philadelphia in the 19th century....
4.  The magazine was started in 1989....
5.  The magazine was started in 1989....
[DEBUG] Context length: 344
[DEBUG] QA output: {'score': 0.04971929267048836, 'start': 0, 'end': 15, 'answer': 'First for Women'}

[PREDICTED] First for Women
[F1 SCORE] 0.0

[TEST CASE 2]
[QUESTION] The Oberoi family is pa

In [ ]:
# ==============================
# 1. LOAD DATASET
# ==============================
from datasets import load_dataset

print("\n[STEP 1] Loading dataset...")
dataset = load_dataset("hotpot_qa", "fullwiki", split="train[:100]")

print("[INFO] Dataset size:", len(dataset))
print("[INFO] First question:", dataset[0]["question"])
print("[INFO] First answer:", dataset[0]["answer"])
print("------")


# ==============================
# 2. GET FULL CONTEXT (NO FILTER)
# ==============================
print("\n[STEP 2] Using FULL context (no supporting facts)...")

def get_context(sample):
    context_list = []

    for title, sentences in zip(sample["context"]["title"], sample["context"]["sentences"]):
        context_list.extend(sentences)

    return context_list


sample_context = get_context(dataset[0])
print("[DEBUG] Total context sentences:", len(sample_context))
print("[DEBUG] Example context:", sample_context[:2])
print("------")


# ==============================
# 3. CHUNKING
# ==============================
print("\n[STEP 3] Chunking text...")

def chunk_text(text):
    return [text]  # sentence-level chunks

print("[INFO] Chunking method: sentence-level")
print("------")


# ==============================
# 4. BUILD DOCUMENT STORE
# ==============================
print("\n[STEP 4] Building document store...")

docs = []

for idx, sample in enumerate(dataset):
    contexts = get_context(sample)
    for c in contexts:
        docs.extend(chunk_text(c))

    if idx % 20 == 0:
        print(f"[PROGRESS] Processed {idx} samples")

print("[INFO] Total documents:", len(docs))
print("[DEBUG] Sample doc:", docs[0][:100])
print("------")


# ==============================
# 5. EMBEDDINGS
# ==============================
print("\n[STEP 5] Encoding documents...")

from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(docs, normalize_embeddings=True)

print("[INFO] Embedding dimension:", embeddings.shape[1])
print("[INFO] Total embeddings:", embeddings.shape[0])
print("------")


# ==============================
# 6. FAISS INDEX
# ==============================
print("\n[STEP 6] Building FAISS index...")

import faiss
import numpy as np

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(np.array(embeddings))

print("[INFO] FAISS index size:", index.ntotal)
print("------")


# ==============================
# 7. LOAD RERANKER
# ==============================
print("\n[STEP 7] Loading reranker...")

from sentence_transformers import CrossEncoder

reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

print("[INFO] Reranker loaded")
print("------")


# ==============================
# 8. RETRIEVE + RERANK
# ==============================
print("\n[STEP 8] Defining retrieval...")

def retrieve(query, k=5):
    print("\n[RETRIEVAL DEBUG]")
    print("Query:", query)

    # Step 1: retrieve top 20 candidates
    q_emb = model.encode([query], normalize_embeddings=True)
    distances, indices = index.search(q_emb, 20)

    candidate_docs = [docs[i] for i in indices[0]]

    print("[DEBUG] Retrieved candidates:", len(candidate_docs))

    # Step 2: reranking
    pairs = [(query, doc) for doc in candidate_docs]
    scores = reranker.predict(pairs)

    ranked = sorted(zip(candidate_docs, scores), key=lambda x: x[1], reverse=True)

    # Step 3: select top-k
    top_docs = [doc for doc, _ in ranked[:k]]

    print("[DEBUG] Top docs after reranking:")
    for i, doc in enumerate(top_docs):
        print(f"{i+1}. {doc[:120]}...")

    return top_docs


# ==============================
# 9. QA MODEL
# ==============================
print("\n[STEP 9] Loading QA model...")

from transformers import pipeline

qa_pipeline = pipeline(
    "question-answering",
    model="deepset/roberta-base-squad2"
)

print("[INFO] QA model loaded")
print("------")


# ==============================
# 10. ANSWER FUNCTION
# ==============================
print("\n[STEP 10] Defining answer function...")

def answer_question(question):
    contexts = retrieve(question, k=5)
    context = " ".join(contexts)

    print("[DEBUG] Context length:", len(context))

    result = qa_pipeline(question=question, context=context)

    print("[DEBUG] QA output:", result)

    return result["answer"]


# ==============================
# 11. NORMALIZATION
# ==============================
def normalize(text):
    return text.lower().strip()


# ==============================
# 12. F1 SCORE
# ==============================
def compute_f1(pred, gt):
    pred_tokens = normalize(pred).split()
    gt_tokens = normalize(gt).split()

    common = set(pred_tokens) & set(gt_tokens)

    if len(common) == 0:
        return 0.0

    precision = len(common) / len(pred_tokens)
    recall = len(common) / len(gt_tokens)

    return 2 * (precision * recall) / (precision + recall)


# ==============================
# 13. RUN EVALUATION
# ==============================
print("\n[STEP 13] Running evaluation...\n")

total_f1 = 0

for i in range(5):
    print("\n==============================")
    print(f"[TEST CASE {i+1}]")

    q = dataset[i]["question"]
    gt = dataset[i]["answer"]

    print("[QUESTION]", q)
    print("[GROUND TRUTH]", gt)

    pred = answer_question(q)

    f1 = compute_f1(pred, gt)
    total_f1 += f1

    print("\n[PREDICTED]", pred)
    print("[F1 SCORE]", round(f1, 3))
    print("==============================")

avg_f1 = total_f1 / 5
print("\n🔥 FINAL AVERAGE F1:", round(avg_f1, 3))


[STEP 1] Loading dataset...
[INFO] Dataset size: 100
[INFO] First question: Which magazine was started first Arthur's Magazine or First for Women?
[INFO] First answer: Arthur's Magazine
------

[STEP 2] Using FULL context (no supporting facts)...
[DEBUG] Total context sentences: 46
[DEBUG] Example context: ["Radio City is India's first private FM radio station and was started on 3 July 2001.", ' It broadcasts on 91.1 (earlier 91.0 in most cities) megahertz from Mumbai (where it was started in 2004), Bengaluru (started first in 2001), Lucknow and New Delhi (since 2003).']
------

[STEP 3] Chunking text...
[INFO] Chunking method: sentence-level
------

[STEP 4] Building document store...
[PROGRESS] Processed 0 samples
[PROGRESS] Processed 20 samples
[PROGRESS] Processed 40 samples
[PROGRESS] Processed 60 samples
[PROGRESS] Processed 80 samples
[INFO] Total documents: 4261
[DEBUG] Sample doc: Radio City is India's first private FM radio station and was started on 3 July 2001.
------

[ST

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[INFO] Embedding dimension: 384
[INFO] Total embeddings: 4261
------

[STEP 6] Building FAISS index...
[INFO] FAISS index size: 4261
------

[STEP 7] Loading reranker...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[INFO] Reranker loaded
------

[STEP 8] Defining retrieval...

[STEP 9] Loading QA model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaForQuestionAnswering LOAD REPORT from: deepset/roberta-base-squad2
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[INFO] QA model loaded
------

[STEP 10] Defining answer function...

[STEP 13] Running evaluation...


[TEST CASE 1]
[QUESTION] Which magazine was started first Arthur's Magazine or First for Women?
[GROUND TRUTH] Arthur's Magazine

[RETRIEVAL DEBUG]
Query: Which magazine was started first Arthur's Magazine or First for Women?
[DEBUG] Retrieved candidates: 20
[DEBUG] Top docs after reranking:
1. First for Women is a woman's magazine published by Bauer Media Group in the USA....
2. First for Women is a woman's magazine published by Bauer Media Group in the USA....
3. Arthur's Magazine (1844–1846) was an American literary periodical published in Philadelphia in the 19th century....
4. In 1898 the first women's magazine was published in China....
5. The first women's magazine was published in Malaysia in 1932....
[DEBUG] Context length: 395
[DEBUG] QA output: {'score': 0.03742641396820545, 'start': 0, 'end': 15, 'answer': 'First for Women'}

[PREDICTED] First for Women
[F1 SCORE] 0.0

[T

In [ ]:
# ==============================
# FULL RAG PIPELINE (ONE CELL)
# ==============================

from datasets import load_dataset
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import pipeline
import faiss
import numpy as np

print("\n[STEP 1] Loading dataset...")
dataset = load_dataset("hotpot_qa", "fullwiki", split="train[:1000]")

print("[INFO] Dataset size:", len(dataset))


# ==============================
# STEP 2: EXTRACT LIMITED WIKI PAGES
# ==============================
print("\n[STEP 2] Selecting limited Wikipedia pages...")

selected_titles = set()

for sample in dataset:
    for title in sample["supporting_facts"]["title"]:
        selected_titles.add(title)

MAX_DOCS = 1000
selected_titles = list(selected_titles)[:MAX_DOCS]

print("[INFO] Selected docs:", len(selected_titles))


# ==============================
# STEP 3: BUILD KB + CHUNKING
# ==============================
print("\n[STEP 3] Building KB with chunking...")

def chunk_text(text, chunk_size=512):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size):
        chunks.append(" ".join(words[i:i+chunk_size]))
    return chunks

docs = []

for sample in dataset:
    for title, sentences in zip(sample["context"]["title"], sample["context"]["sentences"]):
        if title in selected_titles:
            full_text = " ".join(sentences)
            chunks = chunk_text(full_text)
            docs.extend(chunks)

print("[INFO] Total chunks:", len(docs))
print("[DEBUG] Sample chunk:", docs[0][:1000])


# ==============================
# STEP 4: EMBEDDINGS
# ==============================
print("\n[STEP 4] Encoding embeddings...")

embed_model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = embed_model.encode(docs, normalize_embeddings=True)

print("[INFO] Embedding shape:", embeddings.shape)


# ==============================
# STEP 5: FAISS INDEX
# ==============================
print("\n[STEP 5] Building FAISS index...")

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(np.array(embeddings))

print("[INFO] Index size:", index.ntotal)


# ==============================
# STEP 6: RERANKER
# ==============================
print("\n[STEP 6] Loading reranker...")

reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')


# ==============================
# STEP 7: RETRIEVAL
# ==============================
def retrieve(query, k=2):
    print("\n[RETRIEVAL DEBUG]")
    print("Query:", query)

    q_emb = embed_model.encode([query], normalize_embeddings=True)
    distances, indices = index.search(q_emb, 20)

    candidates = [docs[i] for i in indices[0]]

    # Rerank
    pairs = [(query, doc) for doc in candidates]
    scores = reranker.predict(pairs)

    ranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
    top_docs = [doc for doc, _ in ranked[:k]]

    for i, doc in enumerate(top_docs):
        print(f"{i+1}. {doc[:120]}...")

    return top_docs


# ==============================
# STEP 8: QA MODEL
# ==============================
print("\n[STEP 8] Loading QA model...")

qa_model = pipeline("question-answering", model="deepset/roberta-base-squad2")


# ==============================
# STEP 9: ANSWER FUNCTION
# ==============================
def answer_question(question):
    contexts = retrieve(question)
    context = " ".join(contexts)

    result = qa_model(question=question, context=context)
    return result["answer"]


# ==============================
# STEP 10: F1 SCORE
# ==============================
def normalize(text):
    return text.lower().strip()

def compute_f1(pred, gt):
    pred_tokens = normalize(pred).split()
    gt_tokens = normalize(gt).split()

    common = set(pred_tokens) & set(gt_tokens)
    if len(common) == 0:
        return 0.0

    precision = len(common) / len(pred_tokens)
    recall = len(common) / len(gt_tokens)

    return 2 * (precision * recall) / (precision + recall)


# ==============================
# STEP 11: EVALUATION
# ==============================
print("\n[STEP 11] Running evaluation...\n")

total_f1 = 0

for i in range(20):
    print("\n==============================")
    print(f"[TEST CASE {i+1}]")

    q = dataset[i]["question"]
    gt = dataset[i]["answer"]

    print("[QUESTION]", q)
    print("[GROUND TRUTH]", gt)

    pred = answer_question(q)
    f1 = compute_f1(pred, gt)

    total_f1 += f1

    print("[PREDICTED]", pred)
    print("[F1]", round(f1, 3))

avg_f1 = total_f1 / 20
print("\n🔥 FINAL AVG F1:", round(avg_f1, 3))


[STEP 1] Loading dataset...
[INFO] Dataset size: 1000

[STEP 2] Selecting limited Wikipedia pages...
[INFO] Selected docs: 1000

[STEP 3] Building KB with chunking...
[INFO] Total chunks: 1024
[DEBUG] Sample chunk: James Henry Miller (25 January 1915 – 22 October 1989), better known by his stage name Ewan MacColl, was an English folk singer, songwriter, communist, labour activist, actor, poet, playwright and record producer.

[STEP 4] Encoding embeddings...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[INFO] Embedding shape: (1024, 384)

[STEP 5] Building FAISS index...
[INFO] Index size: 1024

[STEP 6] Loading reranker...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



[STEP 8] Loading QA model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaForQuestionAnswering LOAD REPORT from: deepset/roberta-base-squad2
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



[STEP 11] Running evaluation...


[TEST CASE 1]
[QUESTION] Which magazine was started first Arthur's Magazine or First for Women?
[GROUND TRUTH] Arthur's Magazine

[RETRIEVAL DEBUG]
Query: Which magazine was started first Arthur's Magazine or First for Women?
1. Jane was an American magazine created to appeal to the women who grew up reading "Sassy Magazine"; Jane Pratt was the fo...
[PREDICTED] Sassy Magazine
[F1] 0.5

[TEST CASE 2]
[QUESTION] The Oberoi family is part of a hotel company that has a head office in what city?
[GROUND TRUTH] Delhi

[RETRIEVAL DEBUG]
Query: The Oberoi family is part of a hotel company that has a head office in what city?
1. Mirabeau B.V. is a digital agency headquartered in Amsterdam, Netherlands. Mirabeau has offices also in Eindhoven, and R...
[PREDICTED] Paris
[F1] 0.0

[TEST CASE 3]
[QUESTION] Musician and satirist Allie Goertz wrote a song about the "The Simpsons" character Milhouse, who Matt Groening named after who?
[GROUND TRUTH] President Richar

In [ ]:
# ==============================
# FULL RAG PIPELINE (BGE VERSION)
# ==============================

from datasets import load_dataset
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import pipeline
import faiss
import numpy as np

print("\n[STEP 1] Loading dataset...")
dataset = load_dataset("hotpot_qa", "fullwiki", split="train[:100]")
print("[INFO] Dataset size:", len(dataset))


# ==============================
# STEP 2: SELECT LIMITED WIKI
# ==============================
print("\n[STEP 2] Selecting limited Wikipedia pages...")

selected_titles = set()

for sample in dataset:
    for title in sample["supporting_facts"]["title"]:
        selected_titles.add(title)

MAX_DOCS = 100
selected_titles = list(selected_titles)[:MAX_DOCS]

print("[INFO] Selected docs:", len(selected_titles))


# ==============================
# STEP 3: BUILD KB + CHUNKING
# ==============================
print("\n[STEP 3] Building KB with chunking...")

def chunk_text(text, chunk_size=200):  # smaller chunks better for BGE
    words = text.split()
    return [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

docs = []

for sample in dataset:
    for title, sentences in zip(sample["context"]["title"], sample["context"]["sentences"]):
        if title in selected_titles:
            full_text = " ".join(sentences)
            chunks = chunk_text(full_text)

            # 🔥 BGE requires "passage:"
            chunks = [f"passage: {c}" for c in chunks]

            docs.extend(chunks)

print("[INFO] Total chunks:", len(docs))
print("[DEBUG] Sample chunk:", docs[0][:300])


# ==============================
# STEP 4: BGE EMBEDDINGS
# ==============================
print("\n[STEP 4] Encoding embeddings (BGE)...")

embed_model = SentenceTransformer("BAAI/bge-base-en-v1.5")

embeddings = embed_model.encode(
    docs,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("[INFO] Embedding shape:", embeddings.shape)


# ==============================
# STEP 5: FAISS INDEX
# ==============================
print("\n[STEP 5] Building FAISS index...")

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(np.array(embeddings))

print("[INFO] Index size:", index.ntotal)


# ==============================
# STEP 6: RERANKER (KEEP SAME)
# ==============================
print("\n[STEP 6] Loading reranker...")

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")


# ==============================
# STEP 7: RETRIEVAL (FIXED FOR BGE)
# ==============================
def retrieve(query, k=5):
    print("\n[RETRIEVAL DEBUG]")
    print("Query:", query)

    # 🔥 BGE query format
    query = f"query: {query}"

    q_emb = embed_model.encode([query], normalize_embeddings=True)
    distances, indices = index.search(q_emb, 20)

    candidates = [docs[i] for i in indices[0]]

    # remove "passage:" before reranker (important)
    clean_candidates = [c.replace("passage: ", "") for c in candidates]

    # Rerank
    pairs = [(query, doc) for doc in clean_candidates]
    scores = reranker.predict(pairs)

    ranked = sorted(zip(clean_candidates, scores), key=lambda x: x[1], reverse=True)
    top_docs = [doc for doc, _ in ranked[:k]]

    for i, doc in enumerate(top_docs):
        print(f"{i+1}. {doc[:120]}...")

    return top_docs


# ==============================
# STEP 8: QA MODEL
# ==============================
print("\n[STEP 8] Loading QA model...")

qa_model = pipeline("question-answering", model="deepset/roberta-base-squad2")


# ==============================
# STEP 9: ANSWER FUNCTION
# ==============================
def answer_question(question):
    contexts = retrieve(question)
    context = " ".join(contexts)

    result = qa_model(question=question, context=context)
    return result["answer"]


# ==============================
# STEP 10: F1 SCORE
# ==============================
def normalize(text):
    return text.lower().strip()

def compute_f1(pred, gt):
    pred_tokens = normalize(pred).split()
    gt_tokens = normalize(gt).split()

    common = set(pred_tokens) & set(gt_tokens)
    if len(common) == 0:
        return 0.0

    precision = len(common) / len(pred_tokens)
    recall = len(common) / len(gt_tokens)

    return 2 * (precision * recall) / (precision + recall)


# ==============================
# STEP 11: EVALUATION
# ==============================
print("\n[STEP 11] Running evaluation...\n")

total_f1 = 0

for i in range(20):
    print("\n==============================")
    print(f"[TEST CASE {i+1}]")

    q = dataset[i]["question"]
    gt = dataset[i]["answer"]

    print("[QUESTION]", q)
    print("[GROUND TRUTH]", gt)

    pred = answer_question(q)
    f1 = compute_f1(pred, gt)

    total_f1 += f1

    print("[PREDICTED]", pred)
    print("[F1]", round(f1, 3))

avg_f1 = total_f1 / 20
print("\n🔥 FINAL AVG F1:", round(avg_f1, 3))


[STEP 1] Loading dataset...
[INFO] Dataset size: 100

[STEP 2] Selecting limited Wikipedia pages...
[INFO] Selected docs: 100

[STEP 3] Building KB with chunking...
[INFO] Total chunks: 102
[DEBUG] Sample chunk: passage: Arthur's Magazine (1844–1846) was an American literary periodical published in Philadelphia in the 19th century. Edited by T.S. Arthur, it featured work by Edgar A. Poe, J.H. Ingraham, Sarah Josepha Hale, Thomas G. Spear, and others. In May 1846 it was merged into "Godey's Lady's Book".

[STEP 4] Encoding embeddings (BGE)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

[INFO] Embedding shape: (102, 768)

[STEP 5] Building FAISS index...
[INFO] Index size: 102

[STEP 6] Loading reranker...


...

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



[STEP 8] Loading QA model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaForQuestionAnswering LOAD REPORT from: deepset/roberta-base-squad2
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



[STEP 11] Running evaluation...


[TEST CASE 1]
[QUESTION] Which magazine was started first Arthur's Magazine or First for Women?
[GROUND TRUTH] Arthur's Magazine

[RETRIEVAL DEBUG]
Query: Which magazine was started first Arthur's Magazine or First for Women?
1. First for Women is a woman's magazine published by Bauer Media Group in the USA. The magazine was started in 1989. It is...
2. First for Women is a woman's magazine published by Bauer Media Group in the USA. The magazine was started in 1989. It is...
3. Arthur's Magazine (1844–1846) was an American literary periodical published in Philadelphia in the 19th century. Edited ...
4. Jane was an American magazine created to appeal to the women who grew up reading "Sassy Magazine"; Jane Pratt was the fo...
5. Bizarre was a British alternative magazine published from 1997 to 2015. It was published by Dennis Publishing, and was a...
[PREDICTED] First for Women
[F1] 0.0

[TEST CASE 2]
[QUESTION] The Oberoi family is part of a hotel comp

In [ ]:
# ==============================
# FINAL RAG WITH DOCUMENT DIVERSITY
# ==============================

from datasets import load_dataset
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import faiss
import numpy as np
import torch

# ==============================
# STEP 1: LOAD DATA
# ==============================
print("\n[STEP 1] Loading dataset...")
dataset = load_dataset("hotpot_qa", "fullwiki", split="train")
print("[INFO] Dataset size:", len(dataset))


# ==============================
# STEP 2: SELECT WIKI PAGES
# ==============================
print("\n[STEP 2] Selecting Wikipedia pages...")

selected_titles = set()
for sample in dataset:
    for t in sample["supporting_facts"]["title"]:
        selected_titles.add(t)

selected_titles = list(selected_titles)[:105570]
print("[INFO] Selected docs:", len(selected_titles))


# ==============================
# STEP 3: BUILD KB (WITH TITLES)
# ==============================
print("\n[STEP 3] Building KB with titles...")

print("\n[STEP 3] Building KB with titles (optimized)...")

def chunk_text(text, chunk_size=150):
    words = text.split()
    return [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

docs = []
title_to_text = {}

# STEP 1: Collect unique titles with text
for sample in dataset:
    for title, sentences in zip(sample["context"]["title"], sample["context"]["sentences"]):
        if title in selected_titles and title not in title_to_text:
            title_to_text[title] = " ".join(sentences)

print("[INFO] Unique titles collected:", len(title_to_text))


# STEP 2: Chunk each title ONLY ONCE
for title, text in title_to_text.items():
    chunks = chunk_text(text)
    for c in chunks:
        docs.append((title, f"passage: {c}"))

print("[INFO] Total chunks:", len(docs))

print("[INFO] Total chunks:", len(docs))


# ==============================
# STEP 4: EMBEDDINGS
# ==============================
print("\n[STEP 4] Encoding embeddings...")

embed_model = SentenceTransformer("BAAI/bge-base-en-v1.5")

texts = [c for _, c in docs]

embeddings = embed_model.encode(
    texts,
    batch_size=32,
    normalize_embeddings=True,
    show_progress_bar=True
)

dim = embeddings.shape[1]


# ==============================
# STEP 5: FAISS INDEX
# ==============================
print("\n[STEP 5] Building FAISS index...")

index = faiss.IndexFlatIP(dim)
index.add(np.array(embeddings))


# ==============================
# STEP 6: RERANKER
# ==============================
print("\n[STEP 6] Loading reranker...")

reranker = CrossEncoder("BAAI/bge-reranker-base")


# ==============================
# STEP 7: GENERATOR
# ==============================
print("\n[STEP 7] Loading FLAN-T5...")

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)


# ==============================
# STEP 8: DIVERSITY RETRIEVAL 🔥
# ==============================
def retrieve(query, k=3):

    query_bge = f"query: {query}"

    q_emb = embed_model.encode([query_bge], normalize_embeddings=True)
    _, idx = index.search(q_emb, 40)

    candidates = [docs[i] for i in idx[0]]

    # clean text
    clean = [(title, c.replace("passage: ", "")) for title, c in candidates]

    # rerank
    pairs = [(query, text) for _, text in clean]
    scores = reranker.predict(pairs)

    ranked = sorted(zip(clean, scores), key=lambda x: x[1], reverse=True)

    # 🔥 enforce DIFFERENT TITLES
    selected = []
    seen_titles = set()

    for (title, text), score in ranked:
        if title not in seen_titles:
            selected.append(text)
            seen_titles.add(title)

        if len(selected) == k:
            break

    return selected


# ==============================
# STEP 9: GENERATION
# ==============================
def generate_answer(prompt):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=40,
        do_sample=False
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


# ==============================
# STEP 10: ANSWER FUNCTION
# ==============================
def answer_question(question):

    docs = retrieve(question, k=3)

    # 🔥 LIMIT CONTEXT
    context = " ".join(docs)[:800]

    prompt = f"""
Extract the exact answer from the context.

Context:
{context}

Question: {question}

Answer (short, exact):
"""

    return generate_answer(prompt).strip()


# ==============================
# STEP 11: F1 SCORE
# ==============================
def normalize(text):
    return text.lower().strip()

def compute_f1(pred, gt):
    p = normalize(pred).split()
    g = normalize(gt).split()

    common = set(p) & set(g)
    if not common:
        return 0

    precision = len(common)/len(p)
    recall = len(common)/len(g)

    return 2 * precision * recall / (precision + recall)


# ==============================
# STEP 12: EVALUATION
# ==============================
print("\n[STEP 12] Evaluation...\n")

total_f1 = 0

for i in range(500):
    print("\n====================")

    q = dataset[i]["question"]
    gt = dataset[i]["answer"]

    print("[Q]", q)
    print("[GT]", gt)

    pred = answer_question(q)
    f1 = compute_f1(pred, gt)

    total_f1 += f1

    print("[PRED]", pred)
    print("[F1]", round(f1, 3))

print("\n🔥 FINAL F1:", round(total_f1/500, 3))


[STEP 1] Loading dataset...
[INFO] Dataset size: 90447

[STEP 2] Selecting Wikipedia pages...
[INFO] Selected docs: 105570

[STEP 3] Building KB with titles...

[STEP 3] Building KB with titles (optimized)...
[INFO] Unique titles collected: 105570
[INFO] Total chunks: 108890
[INFO] Total chunks: 108890

[STEP 4] Encoding embeddings...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/3403 [00:00<?, ?it/s]


[STEP 5] Building FAISS index...

[STEP 6] Loading reranker...


...

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



[STEP 7] Loading FLAN-T5...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning



[STEP 12] Evaluation...


[Q] Which magazine was started first Arthur's Magazine or First for Women?
[GT] Arthur's Magazine
[PRED] Arthur's Magazine
[F1] 1.0

[Q] The Oberoi family is part of a hotel company that has a head office in what city?
[GT] Delhi
[PRED] Delhi
[F1] 1.0

[Q] Musician and satirist Allie Goertz wrote a song about the "The Simpsons" character Milhouse, who Matt Groening named after who?
[GT] President Richard Nixon
[PRED] President Richard Nixon
[F1] 1.0

[Q]  What nationality was James Henry Miller's wife?
[GT] American
[PRED] American
[F1] 1.0

[Q] Cadmium Chloride is slightly soluble in this chemical, it is also called what?
[GT] alcohol
[PRED] alcohol
[F1] 1.0

[Q] Which tennis player won more Grand Slam titles, Henri Leconte or Jonathan Stark?
[GT] Jonathan Stark
[PRED] Jonathan Stark
[F1] 1.0

[Q] Which genus of moth in the world's seventh-largest country contains only one species?
[GT] Crambidae
[PRED] Asarina
[F1] 0

[Q] Who was once considered the best ki

In [ ]:
 # ==============================
# FINAL RAG WITH DOCUMENT DIVERSITY
# ==============================

from datasets import load_dataset
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import faiss
import numpy as np
import torch

# ==============================
# STEP 1: LOAD DATA
# ==============================
print("\n[STEP 1] Loading dataset...")
dataset = load_dataset("hotpot_qa", "fullwiki", split="train[:500]")
print("[INFO] Dataset size:", len(dataset))


# ==============================
# STEP 2: SELECT WIKI PAGES
# ==============================
print("\n[STEP 2] Selecting Wikipedia pages...")

selected_titles = set()
for sample in dataset:
    for t in sample["supporting_facts"]["title"]:
        selected_titles.add(t)

selected_titles = list(selected_titles)[:105570]
print("[INFO] Selected docs:", len(selected_titles))


# ==============================
# STEP 3: BUILD KB (WITH TITLES)
# ==============================
print("\n[STEP 3] Building KB with titles...")

print("\n[STEP 3] Building KB with titles (optimized)...")

def chunk_text(text, chunk_size=150):
    words = text.split()
    return [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

docs = []
title_to_text = {}

# STEP 1: Collect unique titles with text
for sample in dataset:
    for title, sentences in zip(sample["context"]["title"], sample["context"]["sentences"]):
        if title in selected_titles and title not in title_to_text:
            title_to_text[title] = " ".join(sentences)

print("[INFO] Unique titles collected:", len(title_to_text))


# STEP 2: Chunk each title ONLY ONCE
for title, text in title_to_text.items():
    chunks = chunk_text(text)
    for c in chunks:
        docs.append((title, f"passage: {c}"))

print("[INFO] Total chunks:", len(docs))

print("[INFO] Total chunks:", len(docs))


# ==============================
# STEP 4: EMBEDDINGS
# ==============================
print("\n[STEP 4] Encoding embeddings...")

embed_model = SentenceTransformer("BAAI/bge-base-en-v1.5")

texts = [c for _, c in docs]

embeddings = embed_model.encode(
    texts,
    batch_size=32,
    normalize_embeddings=True,
    show_progress_bar=True
)

dim = embeddings.shape[1]


# ==============================
# STEP 5: FAISS INDEX
# ==============================
print("\n[STEP 5] Building FAISS index...")

index = faiss.IndexFlatIP(dim)
index.add(np.array(embeddings))


# ==============================
# STEP 6: RERANKER
# ==============================
print("\n[STEP 6] Loading reranker...")

reranker = CrossEncoder("BAAI/bge-reranker-base")


# ==============================
# STEP 7: GENERATOR
# ==============================
print("\n[STEP 7] Loading FLAN-T5...")

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-large")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-large")

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)


# ==============================
# STEP 8: DIVERSITY RETRIEVAL 🔥
# ==============================
def retrieve(query, k=3):

    query_bge = f"query: {query}"

    q_emb = embed_model.encode([query_bge], normalize_embeddings=True)
    _, idx = index.search(q_emb, 40)

    candidates = [docs[i] for i in idx[0]]

    # clean text
    clean = [(title, c.replace("passage: ", "")) for title, c in candidates]

    # rerank
    pairs = [(query, text) for _, text in clean]
    scores = reranker.predict(pairs)

    ranked = sorted(zip(clean, scores), key=lambda x: x[1], reverse=True)

    # 🔥 enforce DIFFERENT TITLES
    selected = []
    seen_titles = set()

    for (title, text), score in ranked:
        if title not in seen_titles:
            selected.append(text)
            seen_titles.add(title)

        if len(selected) == k:
            break

    return selected


# ==============================
# STEP 9: GENERATION
# ==============================
def generate_answer(prompt):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=40,
        do_sample=False
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)
# ==============================
# BASELINE (LLM ONLY) 🔥
# ==============================
def baseline_answer(question):

    prompt = f"""
Answer the question directly.

Question: {question}

Answer (short, exact):
"""

    return generate_answer(prompt).strip()

# ==============================
# STEP 10: ANSWER FUNCTION
# ==============================
def answer_question(question):

    docs = retrieve(question, k=3)

    # 🔥 LIMIT CONTEXT
    context = " ".join(docs)[:800]

    prompt = f"""
Extract the exact answer from the context.

Context:
{context}

Question: {question}

Answer (short, exact):
"""

    return generate_answer(prompt).strip()


# ==============================
# STEP 11: F1 SCORE
# ==============================
def normalize(text):
    return text.lower().strip()

def compute_f1(pred, gt):
    p = normalize(pred).split()
    g = normalize(gt).split()

    common = set(p) & set(g)
    if not common:
        return 0

    precision = len(common)/len(p)
    recall = len(common)/len(g)

    return 2 * precision * recall / (precision + recall)


# ==============================
# STEP 12: EVALUATION
# ==============================
print("\n[STEP 12] Evaluation...\n")

print("\n[STEP 12] Evaluation (RAG vs BASELINE)...\n")

rag_total_f1 = 0
base_total_f1 = 0

for i in range(50):   # 🔥 reduce first (faster testing)

    print("\n====================")

    q = dataset[i]["question"]
    gt = dataset[i]["answer"]

    print("[Q]", q)
    print("[GT]", gt)

    # RAG
    rag_pred = answer_question(q)
    rag_f1 = compute_f1(rag_pred, gt)

    # BASELINE
    base_pred = baseline_answer(q)
    base_f1 = compute_f1(base_pred, gt)

    rag_total_f1 += rag_f1
    base_total_f1 += base_f1

    print("\n[RAG PRED]", rag_pred)
    print("[RAG F1]", round(rag_f1, 3))

    print("\n[BASE PRED]", base_pred)
    print("[BASE F1]", round(base_f1, 3))


print("\n🔥 FINAL RESULTS")
print("RAG F1:", round(rag_total_f1/50, 3))
print("BASELINE F1:", round(base_total_f1/50, 3))


[STEP 1] Loading dataset...
[INFO] Dataset size: 500

[STEP 2] Selecting Wikipedia pages...
[INFO] Selected docs: 998

[STEP 3] Building KB with titles...

[STEP 3] Building KB with titles (optimized)...
[INFO] Unique titles collected: 998
[INFO] Total chunks: 1025
[INFO] Total chunks: 1025

[STEP 4] Encoding embeddings...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/33 [00:00<?, ?it/s]


[STEP 5] Building FAISS index...

[STEP 6] Loading reranker...


...

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



[STEP 7] Loading FLAN-T5...


Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.



[STEP 12] Evaluation...


[STEP 12] Evaluation (RAG vs BASELINE)...


[Q] Which magazine was started first Arthur's Magazine or First for Women?
[GT] Arthur's Magazine

[RAG PRED] Arthur's Magazine
[RAG F1] 1.0

[BASE PRED] Arthur's Magazine
[BASE F1] 1.0

[Q] The Oberoi family is part of a hotel company that has a head office in what city?
[GT] Delhi

[RAG PRED] Delhi
[RAG F1] 1.0

[BASE PRED] chennai
[BASE F1] 0

[Q] Musician and satirist Allie Goertz wrote a song about the "The Simpsons" character Milhouse, who Matt Groening named after who?
[GT] President Richard Nixon

[RAG PRED] President Richard Nixon
[RAG F1] 1.0

[BASE PRED] mr. taylor
[BASE F1] 0

[Q]  What nationality was James Henry Miller's wife?
[GT] American

[RAG PRED] American
[RAG F1] 1.0

[BASE PRED] british
[BASE F1] 0

[Q] Cadmium Chloride is slightly soluble in this chemical, it is also called what?
[GT] alcohol

[RAG PRED] alcohol
[RAG F1] 1.0

[BASE PRED] ethanol
[BASE F1] 0

[Q] Which tennis player won more Gr